In [ ]:
%%capture pangbank_tutorials_init_logs

conda_command_path = "bin/micromamba"
amrfinder_env_path = "./amrfinder"
pip_command = "pip"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
from shutil import which
from pathlib import Path

conda_command = ""
if True:
    if not conda_command:
        conda_command_path = "bin/micromamba"
        !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj {conda_command_path}
        if not which(conda_command_path):
            raise RuntimeError("Micromamba installation failed")
        conda_command = conda_command_path
    else:
        if not which(conda_command_path):
            raise RuntimeError(f"conda_command_path: '{conda_command_path}' not found")
        conda_command = conda_command_path

    if not Path(amrfinder_env_path).exists():
        !{conda_command} create -y -p {amrfinder_env_path} -c conda-forge -c bioconda ncbi-amrfinderplus

    amrfinder_command = f"{conda_command} run -p {amrfinder_env_path} amrfinder"
    !{pip_command} install git+https://github.com/labgem/PPanGGOLiN.git@output_rgp_to_fams numpy=2
    !{pip_command} install git+https://github.com/labgem/PanGBank-cli.git
else:
    amrfinder_command = f"amrfinder"

!{amrfinder_command} -u

# PanGBank Tutorial: Analyzing AMR Genes in Pangenomes

This tutorial demonstrates how to:
1. Retrieve pangenomes from PanGBank
2. Annotate gene families with AMRFinderPlus
3. Cluster Regions of Genomic Plasticity (RGPs)
4. Analyze the distribution of AMR genes across the pangenome

## Step 1: Download Pangenome from PanGBank

Search for *Escherichia coli* F pangenomes in the GTDB_refseq collection.

In [ ]:
! pangbank search-pangenomes --collection GTDB_refseq --taxon "s__Escherichia coli_F" --download

In [ ]:
! ppanggolin info -p "pangbank/GTDB_refseq_s__Escherichia_coli_F_id616.h5" --content

## Step 2: Extract Gene Family Sequences

Export protein sequences for all gene families from the pangenome file.

In [ ]:
! ppanggolin fasta -p "pangbank/GTDB_refseq_s__Escherichia_coli_F_id616.h5" -f --compress --prot_families all -o families_faa_output


## Step 3: Annotate AMR Genes with AMRFinderPlus

Run AMRFinderPlus to identify antimicrobial resistance genes in the gene families.

In [ ]:
# need to have amrfinder database downloaded with 'amrfinder -u'
! {amrfinder_command} -p "families_faa_output/all_protein_families.faa.gz" --plus --threads 8  -o amrfinder_result.tsv


## Step 4: Cluster RGPs

Group similar Regions of Genomic Plasticity (RGPs) into clusters based on gene content similarity.

In [ ]:
! ppanggolin rgp_cluster --grr_metric max_grr -p "pangbank/GTDB_refseq_s__Escherichia_coli_F_id616.h5"  -o rgp_cluster -f --add_metadata

## Step 5: Export Pangenome Data

Generate output files containing information about spots, RGPs, gene families, partitions, and modules.

In [ ]:
! ppanggolin write_pangenome  --spots --regions --families_tsv --partitions \
                              --regions_families --output ppanggolin_output \
                                --modules --spot_modules  \
                                -p "pangbank/GTDB_refseq_s__Escherichia_coli_F_id616.h5" -f \
                                

In [ ]:
! tree 


---

# Data Analysis: Combining Pangenome and AMR Annotations

## Load Required Libraries

In [ ]:
import pandas as pd
from pathlib import Path
import plotly.express as px

In [ ]:
resistance_annotation_file = "amrfinder_result.tsv"
df_amrfinder = pd.read_csv(resistance_annotation_file, sep="\t")


df_amrfinder["amrfinder_annotation"] = True
df_amrfinder



## Filter AMR Results

Keep only genes with AMR annotations (excluding virulence and stress response genes).

In [ ]:


arm_filter = df_amrfinder['Type'] == "AMR" 
df_amr = df_amrfinder.loc[arm_filter][['Protein id', 'Element symbol', 'Element name', 'Scope', 'Type',
       'Subtype', 'Class', 'Subclass', 'Method',
       'HMM description', 'amrfinder_annotation']]


df_amr


## Assign Pangenome Partitions to AMR Genes

Read partition files (Persistent, Shell, Cloud) and assign each AMR gene family to its partition.

In [ ]:
def parse_partition_file(p_file):
    with open(p_file) as fl:
        return [l.strip() for l in fl]
        

shell_fams = parse_partition_file('ppanggolin_output/partitions/shell.txt')
cloud_fams = parse_partition_file('ppanggolin_output/partitions/cloud.txt')
persistent_fams = parse_partition_file('ppanggolin_output/partitions/persistent.txt')


df_amr.loc[df_amr['Protein id'].isin(persistent_fams), 'partition'] = "Persistent"
df_amr.loc[df_amr['Protein id'].isin(cloud_fams), 'partition'] = "Cloud"
df_amr.loc[df_amr['Protein id'].isin(shell_fams), 'partition'] = "Shell"

df_amr

In [ ]:
partition_to_color = {'Persistent': '#e59c04', 'Shell': '#00d860', 'Cloud': '#79deff'}
partition_order = ["Persistent", "Shell", "Cloud"]



df_amr_rgp_partition_total = df_amr.groupby(["partition"]).agg({
                                    "Protein id":"count"}).reset_index()

fig = px.bar(df_amr_rgp_partition_total, x='partition', y='Protein id', color="partition", 
             color_discrete_map=partition_to_color,
             category_orders={"partition": partition_order},
             title="Total AMR Gene Families per Partition",
             text='Protein id',
             labels={"Protein id": "# Gene Families<br>with AMR annotation", "partition": "Pangenome Partition"})

# Adjust text position to be outside the bars at the top
fig.update_traces(textfont_size=12, width=0.6)

fig.update_layout(
    autosize=False,
    width=700,
    height=450,
)
fig.show()

In [ ]:

df_amr_rgp_partition = df_amr.groupby(["partition", "Class"]).agg({
                                    "Protein id":"count"}).reset_index()


fig = px.bar(df_amr_rgp_partition, x='Class', y='Protein id', color="partition", 
             color_discrete_map=partition_to_color,
             category_orders={"partition": partition_order},
             title="AMR Gene Families by Class and Partition",
             labels={"Protein id": "Number of Gene Families", "Class": "AMR Class"})
fig.show()

In [ ]:
df_amr_persistent =df_amr.loc[df_amr['partition'] == "Persistent"][['Protein id', "partition", "Element symbol", "Element name", "Type", "Class"]]
df_amr_persistent

## Load RGP Clustering Results

Import the RGP cluster assignments generated in the previous step.

In [ ]:
df_rgp_cluster = pd.read_csv('rgp_cluster/rgp_cluster.tsv', sep='\t')

df_rgp_cluster

In [ ]:
# read rgps families
rgp_families_file = Path('ppanggolin_output/rgp_families.tsv')
df_rgp_fams = pd.read_csv(rgp_families_file, sep='\t')
df_rgp_fams



## Load Functional Modules

Import gene family module assignments to identify functional units in RGPs.

In [ ]:
module_families_file = Path("ppanggolin_output/functional_modules.tsv")
df_module_fams = pd.read_csv(module_families_file, sep="\t")
df_module_fams

## Merge All Data Tables

Combine RGP families, cluster assignments, modules, and AMR annotations into a single dataframe for analysis.

In [ ]:
df_rgp_info_merged = df_rgp_fams.merge(df_rgp_cluster, left_on="rgp_id", right_on="RGPs", how="left")
df_rgp_info_merged = df_rgp_info_merged.merge(df_module_fams, on="family_id", how="left")
df_rgp_info_merged = df_rgp_info_merged.merge(df_amr, left_on="family_id", right_on="Protein id", how="left")
df_rgp_info_merged

---

# Spot-Level Analysis

## Calculate Summary Statistics per Spot

Count RGPs, clusters, gene families, and modules for each genomic hotspot (spot).

In [ ]:
# Create spot-level summary with some metrics
# Filter for rows with AMR annotation
has_amr = df_rgp_info_merged['amrfinder_annotation'].notna()


# Overall counts per spot
overall = df_rgp_info_merged.groupby('spot_id').agg(
    n_clusters=('cluster', 'nunique'),
    n_rgps=('RGPs', 'nunique'),
    n_families=('family_id', 'nunique'),
    n_modules=('module_id', 'nunique')
)

# AMR-specific counts per spot
amr_only = df_rgp_info_merged.loc[has_amr].groupby('spot_id').agg(
    n_clusters_with_amr=('cluster', 'nunique'),
    n_rgps_with_amr=('RGPs', 'nunique'),
    n_families_with_amr=('family_id', 'nunique'),
    n_modules_with_amr=('module_id', 'nunique')
)


# Combine into single dataframe
spot_summary = overall.join(amr_only, how='left').fillna(0).astype(int).reset_index()

spot_summary['prct_rgp_with_amr'] = (spot_summary['n_rgps_with_amr'] / spot_summary['n_rgps'] * 100).round(2)
spot_summary['prct_clusters_with_amr'] = (spot_summary['n_clusters_with_amr'] / spot_summary['n_clusters'] * 100).round(2)
spot_summary



In [ ]:


# Scatter: spot size vs % RGPs with AMR, colored by number of AMR families
no_spot_filter = spot_summary["spot_id"] != "No spot"
 
fig = px.scatter(
    spot_summary.loc[no_spot_filter],
    x="n_rgps",
    y="prct_rgp_with_amr",
    hover_data=spot_summary.columns,
    color="n_families_with_amr",
    size="n_families_with_amr",
    title="Spot size vs % RGPs with AMR", # text="spot_of_interest",
    marginal_x="histogram", marginal_y="histogram"
)
fig.update_coloraxes(colorscale="Viridis")

fig.update_layout(
    autosize=False,
    width=1000,
    height=600,
    xaxis_title="Number of RGPs in spot",
    yaxis_title="% RGPs with AMR annotation",
    coloraxis_colorbar=dict(title="# AMR families")
)

# Add axis titles to marginal plots
fig.layout.xaxis2.title.text = "# Spot"

fig.layout.xaxis2.showticklabels = True 


fig.layout.yaxis3.title.text = "# Spot"

fig.layout.yaxis3.showticklabels = True 


fig.show()


---

# RGP Cluster Analysis

## Calculate Summary Statistics per RGP Cluster

Aggregate data by RGP cluster to identify clusters with AMR genes and their distribution across spots.

In [ ]:
has_amr = df_rgp_info_merged['amrfinder_annotation'].notna()
no_spot_filter = df_rgp_info_merged['spot_id'] != "No spot"

# Overall counts per spot
overall = df_rgp_info_merged[no_spot_filter].groupby('cluster').agg(
    n_spots=('spot_id', 'nunique'),
    n_rgps=('RGPs', 'nunique'),
    n_families=('family_id', 'nunique'),
    n_modules=('module_id', 'nunique')
)

# AMR-specific counts per spot
amr_only = df_rgp_info_merged[no_spot_filter & has_amr ].groupby('cluster').agg(
    n_spots_with_amr=('spot_id', 'nunique'),
    n_rgps_with_amr=('RGPs', 'nunique'),
    n_families_with_amr=('family_id', 'nunique'),
    n_modules_with_amr=('module_id', 'nunique'),
)


# Combine into single dataframe
cluster_summary = overall.join(amr_only, how='left').fillna(0).reset_index()



cluster_summary['prct_rgp_with_amr'] = (cluster_summary['n_rgps_with_amr'] / cluster_summary['n_rgps'] * 100).round(2)
cluster_summary['prct_spot_with_amr'] = (cluster_summary['n_spots_with_amr'] / cluster_summary['n_spots'] * 100).round(2)


has_module = df_rgp_info_merged['module_id'].notna()
amr_modules = df_rgp_info_merged[has_module & has_amr].groupby('cluster').agg(
    amr_modules=('module_id', set),
)
amr_modules['amr_modules'] = amr_modules['amr_modules'].apply(lambda x: ' '.join(x) if isinstance(x, set) else x)


amr_spots = df_rgp_info_merged[has_amr & no_spot_filter].groupby('cluster').agg(
    amr_spots=('spot_id', set),
)
amr_spots['amr_spots'] = amr_spots['amr_spots'].apply(lambda x: ' '.join(x) if isinstance(x, set) else x)


cluster_summary = cluster_summary.merge(amr_modules, how='left', on='cluster').merge(amr_spots, how='left', on='cluster')
cluster_summary


In [ ]:
spot_size_filter = cluster_summary['n_spots_with_amr'] > 0
fig = px.scatter(
    cluster_summary.loc[spot_size_filter].sort_values(by='n_spots_with_amr', ascending=False),
    x="cluster",
    y="n_spots_with_amr",
    hover_data=cluster_summary.columns,
    size="n_families_with_amr",
    color="n_families_with_amr",
)
fig.update_traces(marker=dict(line=dict(width=1, color='black')))
fig.show()
